In [1]:
%run shared_imports.py
from decimal import Decimal
from dataclasses import dataclass
from sqlalchemy.orm.query import Query


In [2]:
engine = make_engine("settings.toml")
session = Session(engine)


In [5]:
query = session.query(Feedback.key_name, Feedback.json).join(Round).filter(
    Feedback.key_name == "mindflayer_abilities")

raw_mindflayer_abilities = pd.read_sql_query(query.statement, session.connection())

In [14]:
raw_mindflayer_abilities

,key_name,json
0,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
1,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
2,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
3,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
4,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
5,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
6,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
7,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
8,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...
9,mindflayer_abilities,{'data': {'Self-Augment Operations': {'purchas...


In [10]:
normalized_mindflayer_abilities = pd.json_normalize(raw_mindflayer_abilities.json)

In [16]:
print(normalized_mindflayer_abilities.sum().astype(int).to_csv())

,0
data.Self-Augment Operations.purchased,18
data.Quick Reboot.purchased,18
data.Swarmprod.purchased,18
data.Enhanced Voice Mod.purchased,2
data.Integrated Access Tuner.purchased,3
data.Flesh Facsimile.purchased,2
data.Enhanced Optical Sensitivity.purchased,6
data.Enhanced Optical Sensitivity.upgraded.2,4
data.Heat Sink.purchased,2
data.Fluid Feet.purchased,7
data.Fluid Feet.upgraded.2,3
data.Internal Faraday Cage.purchased,10
data.Internal Faraday Cage.upgraded.2,8
data.Reinforced Joints.purchased,5
data.Traceroute.purchased,2
data.Laser Arm Augmentation.purchased,6
data.Laser Arm Augmentation.upgraded.2,2
data.Pneumatic Flak Gun.purchased,4
data.Integrated Feedback Arm.purchased,7
data.Quicksilver Form.purchased,6
data.Swarm Absorption Efficiency.purchased,5
data.Armored Plating.purchased,6
data.Armored Plating.upgraded.2,4
data.Armored Plating.upgraded.3,3
data.Laser Arm Augmentation.upgraded.3,1
data.Internal Nanite Application.purchased,1
data.Camfecting Bug.purchased,3
data.Insul

In [19]:
pd.json_normalize(raw_mindflayer_abilities.json, 'data')

TypeError: {'data': {'Self-Augment Operations': {'purchased': 1}, 'Quick Reboot': {'purchased': 1}, 'Swarmprod': {'purchased': 1}}} has non list value {'Self-Augment Operations': {'purchased': 1}, 'Quick Reboot': {'purchased': 1}, 'Swarmprod': {'purchased': 1}} for path data. Must be list or null.

In [14]:
def apply_elite_winloss(text):
    l = text['data']
    result = list()
    for k, v in l.items():
        result.append((k, v))
    if len(result):
        keys, values = zip(*result)
        series = pd.Series(values, index=keys)
        # print(series)
        return series
    else:
        return pd.Series()

def get_elite_winloss(df):
    return pd.concat([df, df['json'].apply(apply_elite_winloss)], axis=1).drop(['json'], axis=1)

In [15]:
elite_winloss = get_elite_winloss(raw_elite_winloss)

In [16]:
elite_winloss

,key_name,pandora,herald,legionnaire,goliath broodmother
0,ai_controlled_elite_loss,2.0,NaN,NaN,NaN
1,ai_controlled_elite_win,2.0,NaN,NaN,NaN
2,ai_controlled_elite_loss,NaN,1.0,NaN,NaN
3,ai_controlled_elite_win,NaN,NaN,1.0,NaN
4,player_controlled_elite_win,1.0,NaN,NaN,NaN
...,...,...,...,...,...
555,ai_controlled_elite_win,NaN,NaN,NaN,1.0
556,ai_controlled_elite_win,1.0,NaN,NaN,NaN
557,ai_controlled_elite_win,NaN,NaN,1.0,NaN
558,ai_controlled_elite_loss,1.0,NaN,NaN,NaN


In [19]:
elite_winloss.groupby(['key_name']).sum().astype(int)

,pandora,herald,legionnaire,goliath broodmother
key_name,,,,
ai_controlled_elite_loss,33,48,32,34
ai_controlled_elite_win,56,59,73,68
player_controlled_elite_loss,39,40,29,30
player_controlled_elite_win,19,17,25,40


In [261]:
totals_df.drop(['datetime'], inplace=True, axis=1)

In [268]:
totals_df[list(parts_kits_names)].sum().astype(int).sort_values()

Arc Revolver                  33
Energy Crossbow               47
u-ION Silencer                77
Plasma Pistol                102
Temperature Gun              103
LWAP Laser Sniper            107
Decloner                     109
SPRK-12 Pistol               152
Accelerator Laser Cannon     303
Ion Carbine                  445
Xray Laser Gun               635
Immolator Laser Gun          756
Advanced Energy Gun         1587
dtype: int32